# 03 — Signals on the 2023 collection (Plan Step 4)

Copy of the pilot `HALT_RAG_v5_signals.ipynb`, pointed at the 2023 Wikipedia collection.
The pilot notebook is kept unchanged as the record of the pilot.

**Input** (data folder — `data\` on the laptop, Drive on Colab), both written by
`notebooks/02b_wiki2023_build.ipynb`:

* `wiki2023_cache.pkl` — subject pages + random background pages from the 1 Nov 2023 dump
* `halt_rag_audit_w23.csv` — the questions, with `m2_status_w23`

**Output:** `halt_rag_w23_signals.csv`, `halt_rag_w23_results.json`

### What is different from the pilot notebook

| # | Pilot (v5) | Here |
|---|---|---|
| 1 | KILT 2019 cache (`kilt_cache_v5.pkl`), with a hashed-filename fallback | `wiki2023_cache.pkl`, no fallback |
| 2 | Lift divided by `sd if sd > 0 else 0.0` | `max(sd, eps)`, `eps` from `config.yaml` |
| 3 | `y_cov` read from the `m3_covered` column | recomputed from this notebook's own top-5 passages |
| 4 | `evaluable` from `m2_status_v5` | from `m2_status_w23` (`found` / `answer_not_on_page`), asserted present |
| 5 | Drive path, settings hard-coded in `CONFIG` | every setting read from `config.yaml` |
| 6 | embeddings cached as `corp_embs_v5.npy` | `corp_embs_w23.npy`, so the pilot cache is never overwritten |
| 7 | a cell scoring CAS against the 50 hand labels | removed — those labels were dropped |

### Still to do before the 1,000-question run

**Switch this notebook to `src/halt_rag`** — `merge_into_passages`, `dedupe` and
`embedding.lift` — instead of the inline pilot copies kept below. The inline
`chunk_text` cuts every paragraph on its own and does not remove duplicates.

`y_judge` still uses an invented prompt, not the published Joren et al. one. Do not
report it. `y_cov` is still a string match, so treat it as provisional.

## Cell 1 — Install

In [1]:
import sys
if "google.colab" in sys.modules:
    !pip install -q sentence-transformers==3.0.1 2>&1 | tail -2
else:
    print("laptop: sentence-transformers comes from .venv (requirements.txt)")
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch 2.14.1+cu130 | cuda: True


## Cell 2 — Setup and settings

Every path, seed and size comes from `config.yaml`.

In [2]:
# ── Setup: works on a laptop (VS Code) and on Colab ───────────────────────
import os, sys, json, subprocess
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    # Colab: get the code from GitHub (push your latest commit first) and mount Drive
    REPO, BRANCH = "/content/HALT-RAG", "lift_signal"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Shaithra2107/HALT-RAG", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # laptop: this notebook sits in <repo>/notebooks, so the repo is one level up
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

from halt_rag.config import load_config, data_path, data_dir
cfg = load_config(os.path.join(REPO, "config.yaml"))
print("running on:", "Colab" if ON_COLAB else "laptop", "| data folder:", data_dir(cfg))

# Nothing below is hard-coded: change config.yaml, not this cell.
CONFIG = {
    'audit_csv'    : data_path(cfg, "audit_w23"),          # halt_rag_audit_w23.csv
    'cache_pkl'    : data_path(cfg, "wiki2023_cache"),     # wiki2023_cache.pkl
    'out_signals'  : data_path(cfg, "signals_w23"),        # halt_rag_w23_signals.csv
    'out_results'  : data_path(cfg, "results_w23"),
    'emb_npy'      : data_path(cfg, "corpus_embs_w23"),

    'bi_encoder'   : cfg["retrieval"]["encoder"],
    'qa_model'     : cfg["signals"]["qa_model"],
    'lm'           : cfg["signals"]["lm"],

    'top_k'        : cfg["retrieval"]["top_k"],
    'chunk_words'  : cfg["collection"]["passage_words"],
    'n_distractors': cfg["collection"]["background_pages_used"],
    'lift_pool'    : cfg["signals"]["lift_pool"],
    'lift_eps'     : cfg["signals"]["lift_eps"],
    'max_ans_len'  : cfg["signals"]["cas_max_answer_len"],
    'seed'         : cfg["seed"],

    # load the cached corpus embeddings instead of re-encoding.
    # Set False only if you want to force a fresh encode.
    'reuse_embeddings': True,
}
print(json.dumps(CONFIG, indent=2))

for _k in ('audit_csv', 'cache_pkl'):
    assert os.path.exists(CONFIG[_k]), (
        f"Not found: {CONFIG[_k]}\nRun notebooks/02b_wiki2023_build.ipynb first.")

Mounted at /content/drive
running on: Colab | data folder: /content/drive/MyDrive/halt_rag_cache
{
  "audit_csv": "/content/drive/MyDrive/halt_rag_cache/halt_rag_audit_w23.csv",
  "cache_pkl": "/content/drive/MyDrive/halt_rag_cache/wiki2023_cache.pkl",
  "out_signals": "/content/drive/MyDrive/halt_rag_cache/halt_rag_w23_signals.csv",
  "out_results": "/content/drive/MyDrive/halt_rag_cache/halt_rag_w23_results.json",
  "emb_npy": "/content/drive/MyDrive/halt_rag_cache/corp_embs_w23.npy",
  "bi_encoder": "sentence-transformers/all-MiniLM-L6-v2",
  "qa_model": "deepset/roberta-base-squad2",
  "lm": "google/flan-t5-base",
  "top_k": 5,
  "chunk_words": 100,
  "n_distractors": 5000,
  "lift_pool": 5000,
  "lift_eps": 1e-06,
  "max_ans_len": 30,
  "seed": 42,
  "reuse_embeddings": true
}


## Cell 3 — Load the 2023 collection and rebuild the corpus

`wiki2023_cache.pkl` holds the subject pages and the random background pages, read
straight from the 1 Nov 2023 dump by `02b_wiki2023_build.ipynb`. It uses the same
key names as the old KILT caches (`kilt_pages`, `distractor_pool`), so the code
below did not have to change.

There is no fallback cache any more. If the file is missing, run 02b.

In [3]:
import pickle, re
import pandas as pd, numpy as np

def normalise_title(t): return re.sub(r'_', ' ', str(t).lower().strip())

audit = pd.read_csv(CONFIG['audit_csv'])
print(f"audit CSV : {os.path.basename(CONFIG['audit_csv'])}  ({len(audit)} rows)")

with open(CONFIG['cache_pkl'], 'rb') as f:
    _c = pickle.load(f)
kilt_pages      = _c['kilt_pages']        # same key names as the old KILT caches
distractor_pool = _c['distractor_pool']
print(f"cache     : {os.path.basename(CONFIG['cache_pkl'])}")
print(f"source    : {_c.get('source', 'unknown')}")
print(f"subject pages in cache : {len(kilt_pages)}")
print(f"background pages       : {len(distractor_pool)}")
if len(distractor_pool) < CONFIG['n_distractors']:
    print(f"\n!! cache holds only {len(distractor_pool)} background pages, you asked for "
          f"{CONFIG['n_distractors']}.\n"
          f"   Lift's baseline will be weak. Raise collection.reservoir_pages and re-run 02b.")

def chunk_text(paragraphs, w=CONFIG['chunk_words']):
    out = []
    for para in paragraphs:
        words = str(para).split()
        for i in range(0, len(words), w):
            c = ' '.join(words[i:i+w])
            if len(c.strip()) > 30: out.append(c)
    return out

passages, titles, is_entity = [], [], []
for norm_title, paras in kilt_pages.items():
    for c in chunk_text(paras):
        passages.append(c); titles.append(norm_title); is_entity.append(1)
for d in distractor_pool[:CONFIG['n_distractors']]:
    for c in chunk_text(d['paragraphs']):
        passages.append(c); titles.append(d['title']); is_entity.append(0)

is_entity = np.array(is_entity)
print(f"\ncorpus: {len(passages):,} passages "
      f"({is_entity.sum():,} from subject pages, {(1-is_entity).sum():,} background)")
print("NOTE: this is a pilot-sized corpus, not all of Wikipedia. State that in the write-up.")
print("TODO: use src/halt_rag merge_into_passages + dedupe before the 1,000-question run;")
print("      chunk_text above cuts each paragraph on its own and keeps duplicates.")

audit CSV : halt_rag_audit_w23.csv  (300 rows)
cache     : wiki2023_cache.pkl
source    : wikimedia/wikipedia/20231101.en
subject pages in cache : 298
background pages       : 10000

corpus: 51,174 passages (6,101 from subject pages, 45,073 background)
NOTE: this is a pilot-sized corpus, not all of Wikipedia. State that in the write-up.
TODO: use src/halt_rag merge_into_passages + dedupe before the 1,000-question run;
      chunk_text above cuts each paragraph on its own and keeps duplicates.


## Cell 4 — Embed the corpus, build the index, retrieve top-5

The embeddings are cached to Drive as a `.npy` file, so this only costs you the
full encoding time **once**. On later runs it loads in seconds.

The cache is keyed on the number of passages. If you re-scan with a different
reservoir size the count changes, the cache is treated as stale, and it
re-encodes automatically. Set `reuse_embeddings` to False to force that by hand.

In [4]:
import sys
if "google.colab" in sys.modules:
    !pip install -q faiss-cpu
else:
    print("laptop: faiss-cpu comes from .venv (requirements.txt)")

In [5]:
from sentence_transformers import SentenceTransformer
import faiss

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
bi = SentenceTransformer(CONFIG['bi_encoder'], device=DEV)

# a cache of its own, so the pilot's corp_embs_v5.npy is never overwritten
EMB_NPY  = CONFIG['emb_npy']
EMB_META = CONFIG['emb_npy'] + '.meta.json'

def _meta_ok():
    if not (os.path.exists(EMB_NPY) and os.path.exists(EMB_META)): return False
    try:
        m = json.load(open(EMB_META))
    except Exception:
        return False
    return (m.get('n_passages') == len(passages)
            and m.get('chunk_words') == CONFIG['chunk_words']
            and m.get('bi_encoder') == CONFIG['bi_encoder'])

if CONFIG['reuse_embeddings'] and _meta_ok():
    corp_embs = np.load(EMB_NPY)
    print(f"loaded cached embeddings: {corp_embs.shape}  ({EMB_NPY})")
else:
    if os.path.exists(EMB_NPY):
        print("cached embeddings are stale (corpus or model changed) — re-encoding")
    print(f"encoding {len(passages):,} passages ... this is the one slow part")
    corp_embs = bi.encode(passages, batch_size=256, show_progress_bar=True,
                          normalize_embeddings=True).astype('float32')
    np.save(EMB_NPY, corp_embs)
    json.dump({'n_passages': len(passages),
               'chunk_words': CONFIG['chunk_words'],
               'bi_encoder': CONFIG['bi_encoder'],
               'dim': int(corp_embs.shape[1])}, open(EMB_META, 'w'))
    mb = corp_embs.nbytes / 1e6
    print(f"saved {EMB_NPY}  ({mb:,.0f} MB) — later runs will load this instead")

assert len(corp_embs) == len(passages), \
    f"embeddings ({len(corp_embs)}) do not match passages ({len(passages)})"

index = faiss.IndexFlatIP(corp_embs.shape[1])
index.add(corp_embs)
print(f"index: {index.ntotal:,} vectors, dim={corp_embs.shape[1]}")

# question embeddings are tiny (300 rows), never worth caching
q_emb = bi.encode(audit['question'].tolist(), batch_size=128,
                  show_progress_bar=True, normalize_embeddings=True).astype('float32')
ret_scores, ret_idx = index.search(q_emb, CONFIG['top_k'])
print("retrieved:", ret_idx.shape)

/usr/local/lib/python3.13/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

Batches:   0%|          | 0/200 [00:00<?, ?it/s]

saved /content/drive/MyDrive/halt_rag_cache/corp_embs_w23.npy  (79 MB) — later runs will load this instead
index: 51,174 vectors, dim=384


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

retrieved: (300, 5)


## Cell 5 — Signals 1–3: Lift (fixed), relevance, consistency

**The fix.** `LIFT_POOL` is a random sample of the corpus, taken once. The old
code sampled from the retrieved passages, which are all passages a retriever
already liked — so the bar was set far too high.

Any pool passage that happens to be in a question's own top-5 is dropped for
that question, so Lift never compares a retrieval against itself.

In [6]:
rng = np.random.default_rng(CONFIG['seed'])
EPS = CONFIG['lift_eps']

# FIX 1 — one random sample of the CORPUS, used as the baseline for every question
pool_sz   = min(CONFIG['lift_pool'], len(corp_embs))
pool_idx  = rng.choice(len(corp_embs), size=pool_sz, replace=False)
LIFT_POOL = corp_embs[pool_idx]
pool_set  = {int(j): p for p, j in enumerate(pool_idx)}   # corpus idx -> pool row
print(f"Lift baseline pool: {pool_sz:,} random corpus passages")

lift, relevance, consistency, rnd_mu, rnd_sd = [], [], [], [], []
for i in range(len(audit)):
    qv   = q_emb[i]
    pv   = corp_embs[ret_idx[i]]          # (5, d) the retrieved passages
    sims = pv @ qv
    relevance.append(float(sims.mean()))

    G  = pv @ pv.T
    iu = np.triu_indices(len(pv), k=1)
    consistency.append(float(G[iu].mean()))

    # drop this question's own passages from the baseline, if present
    drop = [pool_set[int(j)] for j in ret_idx[i] if int(j) in pool_set]
    rs   = LIFT_POOL @ qv
    if drop:
        rs = np.delete(rs, drop)

    mu, sd = float(rs.mean()), float(rs.std())
    rnd_mu.append(mu); rnd_sd.append(sd)
    # FIX — max(sd, eps). v5 wrote 'sd if sd > 0 else 0.0', which quietly
    # returned a neutral Lift of 0 whenever the background had no spread.
    lift.append((float(sims.mean()) - mu) / max(sd, EPS))

df = audit.copy()
df['lift']        = lift
df['relevance']   = relevance
df['consistency'] = consistency
df['rand_mu']     = rnd_mu
df['rand_sd']     = rnd_sd
df['ret_idx']     = [list(map(int, r)) for r in ret_idx]
df['ret_text']    = [[passages[int(j)] for j in r] for r in ret_idx]

print(df[['lift','relevance','consistency','rand_mu','rand_sd']].describe().round(3).to_string())
print("\nSanity: rand_mu should now be LOW (random corpus docs are unlike the question).")
print("If rand_mu is close to relevance, the corpus is too small or too narrow.")

Lift baseline pool: 5,000 random corpus passages
          lift  relevance  consistency  rand_mu  rand_sd
count  300.000    300.000      300.000  300.000  300.000
mean     5.373      0.562        0.435    0.065    0.093
std      0.809      0.062        0.113    0.017    0.007
min      3.967      0.371        0.150    0.020    0.070
25%      4.784      0.519        0.347    0.051    0.089
50%      5.151      0.552        0.413    0.068    0.094
75%      5.836      0.602        0.513    0.076    0.097
max      8.089      0.770        0.778    0.099    0.108

Sanity: rand_mu should now be LOW (random corpus docs are unlike the question).
If rand_mu is close to relevance, the corpus is too small or too narrow.


## Cell 6 — Signals 4–5: kw_coverage, clarity (fixed)

**The fix.** `corpus_counter` now counts every passage in the corpus. The old
code counted only the retrieved ones, which turned clarity into "how unusual is
this entity compared to the other 299 entities" — close to a definition of
obscurity, which is exactly how it behaved.

In [7]:
import math
from collections import Counter

STOP = set('''a an the is are was were be been being of in on at to for with by from as and or but
if then than that this these those what which who whom whose when where why how do does did done
it its his her their there here not no nor so such own same too very can will just should now
you your i me my we our they them he she him'''.split())

def toks(s):   return re.findall(r"[a-z0-9]+", str(s).lower())
def content(w): return [x for x in w if x not in STOP and len(x) > 2]

# FIX 2 — background language model over the WHOLE corpus
print("building corpus language model ...")
corpus_counter = Counter()
for t in passages:
    corpus_counter.update(content(toks(t)))
TOT = sum(corpus_counter.values())
print(f"corpus LM: {len(corpus_counter):,} unique terms, {TOT:,} tokens")

def kw_coverage(q, ps):
    kws = set(content(toks(q)))
    if not kws: return 0.0
    return len(kws & set(toks(' '.join(ps)))) / len(kws)

def clarity(ps, smooth=1e-9):
    c = Counter(content(toks(' '.join(ps)))); n = sum(c.values())
    if n == 0: return 0.0
    return float(sum((f/n) * math.log((f/n) / (corpus_counter.get(w, 0)/TOT or smooth))
                     for w, f in c.items()))

df['kw_coverage'] = [kw_coverage(q, ps) for q, ps in zip(df['question'], df['ret_text'])]
df['clarity']     = [clarity(ps) for ps in df['ret_text']]
print(df[['kw_coverage','clarity']].describe().round(3).to_string())

building corpus language model ...
corpus LM: 99,322 unique terms, 1,416,602 tokens
       kw_coverage  clarity
count      300.000  300.000
mean         0.697    5.078
std          0.248    0.734
min          0.000    3.668
25%          0.500    4.553
50%          0.667    5.007
75%          1.000    5.524
max          1.000    7.853


## Cell 7 — Signal 6: CAS (fixed)

**The fix.** The standard SQuAD 2.0 decision compares two numbers:

* `null  = start_logit[CLS] + end_logit[CLS]` — the model's "no answer here" score
* `best  = max over valid spans of (start_logit[a] + end_logit[b])`

`score_diff = null - best`. High means unanswerable. Answerability is `-score_diff`.

v4 instead read the softmax probability at `[CLS]`, which never looks at how
good the best actual span was. Both are computed here so you can compare them,
and the comparison is worth a paragraph in the dissertation.

In [8]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from tqdm.auto import tqdm

qa_tok = AutoTokenizer.from_pretrained(CONFIG['qa_model'])
qa     = AutoModelForQuestionAnswering.from_pretrained(CONFIG['qa_model']).to(DEV).eval()

@torch.no_grad()
def cas_passage_scores(question, ps, max_len=384, topn=20):
    '''Returns (answerability_standard, answerability_softmax_proxy) per passage.'''
    enc = qa_tok([question]*len(ps), list(ps), return_tensors='pt',
                 truncation=True, max_length=max_len, padding=True)
    out = qa(**{k: v.to(DEV) for k, v in enc.items()})
    sl = out.start_logits.detach().cpu().numpy()
    el = out.end_logits.detach().cpu().numpy()

    std, soft = [], []
    for k in range(len(ps)):
        seq = enc.sequence_ids(k)
        ctx = np.array([j for j, s in enumerate(seq) if s == 1])   # passage tokens only
        null = float(sl[k, 0] + el[k, 0])

        if len(ctx) == 0:
            std.append(-1e4); soft.append(0.0); continue

        s_ctx, e_ctx = sl[k, ctx], el[k, ctx]
        best = -1e9
        for a in np.argsort(-s_ctx)[:topn]:
            for b in np.argsort(-e_ctx)[:topn]:
                if b < a or (b - a + 1) > CONFIG['max_ans_len']: continue
                best = max(best, float(s_ctx[a] + e_ctx[b]))
        if best <= -1e8: best = null - 1.0
        std.append(-(null - best))        # higher = more answerable

        # the v4 proxy, kept for comparison
        def sm0(v):
            v = v - v.max(); e = np.exp(v); return float(e[0] / e.sum())
        soft.append(1.0 - math.sqrt(sm0(sl[k]) * sm0(el[k])))

    return np.array(std), np.array(soft)

cas_max, cas_mean, cas_soft_max = [], [], []
for ps in tqdm(df['ret_text'], total=len(df), desc='CAS'):
    ps = [p for p in ps if str(p).strip()]
    a_std, a_soft = cas_passage_scores(df['question'].iloc[len(cas_max)], ps)
    cas_max.append(float(a_std.max()))
    cas_mean.append(float(a_std.mean()))
    cas_soft_max.append(float(a_soft.max()))

df['cas']          = cas_max        # the spec's max-pooled CAS, standard score
df['cas_mean']     = cas_mean       # mean-pooled: less prone to one false positive
df['cas_soft']     = cas_soft_max   # v4's definition, for the comparison
print(df[['cas','cas_mean','cas_soft']].describe().round(3).to_string())
print("\nmax-pooling fires if ANY of 5 passages looks answerable, so it inflates")
print("false positives. Report cas_mean alongside it.")

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

CAS:   0%|          | 0/300 [00:00<?, ?it/s]

           cas  cas_mean  cas_soft
count  300.000   300.000   300.000
mean     7.534    -3.173     0.877
std      5.356     4.350     0.240
min    -11.344   -14.422     0.005
25%      4.607    -6.259     0.911
50%      8.768    -3.110     0.989
75%     11.335    -0.041     0.997
max     16.135     7.934     1.000

max-pooling fires if ANY of 5 passages looks answerable, so it inflates
false positives. Report cas_mean alongside it.


## Cell 8 — Targets and scope flags

Three targets, and they are not equally trustworthy:

* **`y_cov`** — a gold answer string appears in the passages. This is the label
  your own 21-row audit showed is wrong on **57%** of its positives. Treat every
  number computed against it as provisional.
* **`y_gen`** — flan-t5 reads the passages and its answer matches a gold answer.
* **`y_judge`** — flan-t5 is asked whether the passages support the gold answer.
  **The prompt below is not a validated instrument.** Replace it with the
  published autorater prompt from Joren et al. (2025) before reporting anything
  from it.

`evaluable` marks the 136 questions whose gold page the scan actually reached.
For the other 164 the coverage label means "we never looked there".

In [9]:
from transformers import AutoModelForSeq2SeqLM
import torch.nn.functional as F
import unicodedata

lm_tok = AutoTokenizer.from_pretrained(CONFIG['lm'])
lm = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['lm']).to(DEV).eval()

@torch.no_grad()
def seq_logprob(prompt, target):
    e = lm_tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEV)
    l = lm_tok(target, return_tensors='pt', truncation=True, max_length=32).input_ids.to(DEV)
    lg = lm(**e, labels=l).logits
    return float(F.log_softmax(lg, -1).gather(-1, l.unsqueeze(-1)).squeeze(-1).mean())

@torch.no_grad()
def gen(prompt, n=24):
    e = lm_tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEV)
    return lm_tok.decode(lm.generate(**e, max_new_tokens=n, num_beams=1, do_sample=False)[0],
                         skip_special_tokens=True).strip()

P_CTX = "Answer the question using only the passages.\n\nPassages:\n{ctx}\n\nQuestion: {q}\nAnswer:"
J_PLACEHOLDER = ("Passages:\n{ctx}\n\nQuestion: {q}\nProposed answer: {a}\n\n"
                 "Do the passages state or directly imply that this is the answer? "
                 "Answer yes or no.\nAnswer:")

def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode().lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s); s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return ' '.join(s.split())

def hit(pred, golds, short_len=4):
    '''PopQA ships country codes (us, cn, ba) as aliases, so plain substring
    matching makes "us" match inside "business". Short golds need word boundaries.'''
    p = norm(pred)
    if not p: return 0
    for g in golds:
        g = norm(g)
        if not g: continue
        if len(g) <= short_len:
            if re.search(rf'(?<![a-z0-9]){re.escape(g)}(?![a-z0-9])', p): return 1
        elif g == p or g in p or p in g: return 1
    return 0

_T = [("the United States",["United States of America","USA","US"],1),
      ("Praha",["Prague","Praha"],1),
      ("business australia",["US","United States"],0), ("discuss",["US"],0),
      ("it is in the US",["US"],1), ("cnn news",["CN","China"],0),
      ("Paris",["Leeuwarden"],0), ("",["Leeuwarden"],0)]
assert not [t for t in _T if hit(t[0], t[1]) != t[2]], "alias matcher regression"
print(f"alias matcher: {len(_T)}/{len(_T)} ok")

import ast
def parse_answers(x):
    if isinstance(x, str) and x.strip().startswith('['):
        try: return [str(a) for a in ast.literal_eval(x)]
        except Exception: pass
    return [s.strip() for s in str(x).split('|') if s.strip()]
df['answers'] = df['answers_list'].apply(parse_answers)

gens, judges = [], []
for _, r in tqdm(df.iterrows(), total=len(df), desc='targets'):
    ctx  = "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(r['ret_text']) if str(p).strip())
    gold = r['answers'][0] if r['answers'] else ''
    gens.append(gen(P_CTX.format(ctx=ctx, q=r['question'])))
    pr = J_PLACEHOLDER.format(ctx=ctx, q=r['question'], a=gold)
    judges.append(int(seq_logprob(pr, 'yes') > seq_logprob(pr, 'no')))

df['gen_ctx']   = gens
df['y_gen']     = [hit(p, g) for p, g in zip(df['gen_ctx'], df['answers'])]
df['y_judge']   = judges                       # PLACEHOLDER — not validated
# y_cov is recomputed from THIS notebook's own top-5 passages. The old
# m3_covered column is stale: it came from the small pilot index.
# Same string rule as 02b's answer_on_page, so y_cov and m2_status_w23 agree
# on what "the answer appears" means — only the text searched differs.
def cov_norm(t): return re.sub(r'\s+', ' ', str(t).lower().strip())

def answer_in_passages(answers, paras):
    text = cov_norm(' '.join(str(p) for p in paras))
    return int(any(re.search(r'(?<![\w])' + re.escape(cov_norm(a)) + r'(?![\w])', text)
                   for a in answers if cov_norm(a)))

df['y_cov']     = [answer_in_passages(a, ps)
                   for a, ps in zip(df['answers'], df['ret_text'])]
df['is_tail']   = (df['band'].astype(str).str.lower() == 'tail').astype(int)

# The 2023 status column, and only that one. The w23 audit still carries the old
# m2_status_v5, so picking it up by accident would score the 2019 collection.
assert 'm2_status_w23' in df.columns, (
    "halt_rag_audit_w23.csv has no m2_status_w23 column — run 02b_wiki2023_build.ipynb first.")
df['evaluable'] = df['m2_status_w23'].isin(['found', 'answer_not_on_page'])
print("evaluable flag from: m2_status_w23")
print(df["m2_status_w23"].value_counts().to_string())

for t in ['y_cov','y_gen','y_judge']:
    print(f"  {t:<8} positive {int(df[t].sum()):>3}/{len(df)}  ({df[t].mean():.3f})")
print(f"  evaluable (coverage actually known): {int(df['evaluable'].sum())}/{len(df)}")
df.drop(columns=['ret_text']).to_csv(CONFIG['out_signals'], index=False)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

alias matcher: 8/8 ok


targets:   0%|          | 0/300 [00:00<?, ?it/s]

evaluable flag from: m2_status_w23
m2_status_w23
found                 261
answer_not_on_page     38
page_missing            1
  y_cov    positive 218/300  (0.727)
  y_gen    positive 179/300  (0.597)
  y_judge  positive 214/300  (0.713)
  evaluable (coverage actually known): 299/300


## Cell 10 — Per-signal results, with the leak check, on both scopes

`BAND?` means the signal predicts head/tail better than it predicts the target.
Such a signal is detecting entity popularity, not retrieval quality.
`noise_ctrl` is random numbers and must sit near 0.50.

In [11]:
   from sklearn.metrics import roc_auc_score, average_precision_score
   from sklearn.linear_model import LogisticRegression
   from sklearn.model_selection import StratifiedKFold, cross_val_predict
   from sklearn.preprocessing import StandardScaler
   from sklearn.pipeline import make_pipeline

In [13]:
from sklearn.metrics import average_precision_score

SIGNALS = ['lift','relevance','consistency','kw_coverage','clarity','cas']
EXTRA   = ['cas_mean','cas_soft']
df['noise_ctrl'] = np.random.default_rng(CONFIG['seed']).normal(size=len(df))
EVAL = SIGNALS + EXTRA + ['noise_ctrl']

SCOPES = {'all': df, 'evaluable': df[df['evaluable']]}
rows = []
for scope, sub in SCOPES.items():
    it = (sub['is_tail'] == 1).astype(int).values
    for tgt in ['y_cov','y_gen','y_judge']:
        y = sub[tgt].values
        if len(np.unique(y)) < 2:
            print(f"[{scope}/{tgt}] single class — skipped"); continue
        for s in EVAL:
            v = sub[s].values
            a  = roc_auc_score(y, v)
            ab = roc_auc_score(it, v) if len(np.unique(it)) > 1 else 0.5
            rows.append(dict(scope=scope, target=tgt, signal=s, n=len(sub),
                             auroc=a, auroc_flipped=max(a, 1-a),
                             auprc=average_precision_score(y, v),
                             band_auroc=max(ab, 1-ab),
                             leak='BAND?' if max(ab,1-ab) > max(a,1-a) else ''))
res = pd.DataFrame(rows)

for scope in SCOPES:
    for tgt in ['y_cov','y_gen','y_judge']:
        t = res[(res.scope==scope) & (res.target==tgt)]
        if not len(t): continue
        base = SCOPES[scope][tgt].mean()
        print(f"\n=== {scope} / {tgt}  (n={len(SCOPES[scope])}, base rate {base:.3f}) ===")
        print(t.sort_values('auroc_flipped', ascending=False)
               [['signal','auroc','auroc_flipped','auprc','band_auroc','leak']]
               .round(3).to_string(index=False))

for scope, sub in SCOPES.items():
    it = (sub['is_tail']==1).astype(int).values
    print(f"\ncorr(y_cov, is_tail) [{scope}] = {np.corrcoef(sub['y_cov'], it)[0,1]:+.4f}")


=== all / y_cov  (n=300, base rate 0.727) ===
     signal  auroc  auroc_flipped  auprc  band_auroc  leak
   cas_soft  0.785          0.785  0.896       0.557      
        cas  0.782          0.782  0.896       0.560      
       lift  0.658          0.658  0.817       0.611      
   cas_mean  0.643          0.643  0.827       0.655 BAND?
  relevance  0.640          0.640  0.808       0.666 BAND?
kw_coverage  0.617          0.617  0.777       0.666 BAND?
    clarity  0.440          0.560  0.702       0.722 BAND?
consistency  0.551          0.551  0.759       0.662 BAND?
 noise_ctrl  0.492          0.508  0.728       0.508      

=== all / y_gen  (n=300, base rate 0.597) ===
     signal  auroc  auroc_flipped  auprc  band_auroc  leak
   cas_soft  0.714          0.714  0.761       0.557      
        cas  0.711          0.711  0.757       0.560      
       lift  0.608          0.608  0.698       0.611 BAND?
  relevance  0.573          0.573  0.662       0.666 BAND?
    clarity  0.536   

## Cell 11 — SWL and the ablation

Logistic regression over the signals, scored out of fold. This matches what the
literature review claims and keeps one interpretable weight per signal, which is
what RQ2 is actually asking about.

Pick the scope deliberately. `evaluable` is the honest one but has far fewer
negative cases, so the numbers will be noisy — the cell prints the count so you
can judge it.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

SCOPE  = 'all'          # <- switch to 'evaluable'
TARGET = 'y_cov'        # <- switch to 'y_gen' once you trust it

work = SCOPES[SCOPE]
y = work[TARGET].values
print(f"scope={SCOPE}  target={TARGET}  n={len(work)}  "
      f"positives={int(y.sum())}  negatives={int((1-y).sum())}")
if min(int(y.sum()), int((1-y).sum())) < 30:
    print("!! fewer than 30 cases in the smaller class — treat all of this as indicative only.")

def swl(X, y, seed=CONFIG['seed']):
    if len(np.unique(y)) < 2:
        raise ValueError("single-class target — this is what broke the earliest run")
    p = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=seed))
    oof = cross_val_predict(p, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                            method='predict_proba')[:, 1]
    p.fit(X, y)
    return oof, p.named_steps['logisticregression'].coef_[0]

CHEAP = ['lift','relevance','consistency','kw_coverage','clarity']
ALL6  = CHEAP + ['cas']
CONFIGS = {
    'SWL-A  Lift only'            : ['lift'],
    'SWL-B  5 cheap signals'      : CHEAP,
    'SWL-C  6 signals (+CAS)'     : ALL6,
    'SWL-C2 6 signals (cas_mean)' : CHEAP + ['cas_mean'],
    'kw_coverage alone'           : ['kw_coverage'],
    'cas alone'                   : ['cas'],
}
out = []
for name, feats in CONFIGS.items():
    oof, _ = swl(work[feats].values, y)
    out.append(dict(config=name, k=len(feats), auroc=roc_auc_score(y, oof)))

full_oof, W = swl(work[ALL6].values, y)
full = roc_auc_score(y, full_oof)
for s in ALL6:
    keep = [f for f in ALL6 if f != s]
    out.append(dict(config=f'  minus {s}', k=len(keep),
                    auroc=roc_auc_score(y, swl(work[keep].values, y)[0])))

# FIX 7 — this baseline is only meaningful when the target is NOT the string match
if TARGET != 'y_cov':
    out.append(dict(config='string-match baseline', k=1,
                    auroc=roc_auc_score(y, (work['answer_string_match']=='YES').astype(int))))
else:
    print("\n(string-match baseline skipped: y_cov IS the string match, so it would score 1.000)")

o = pd.DataFrame(out); o['delta_vs_6sig'] = (o['auroc'] - full).round(3)
print(); print(o.round(3).to_string(index=False))
print(f"\n6-signal weights: {dict(zip(ALL6, W.round(3)))}")
print("\nIf SWL-C is not clearly above the best single signal, combining buys nothing.")
print("That is a legitimate negative answer to RQ2 — report it.")

scope=all  target=y_cov  n=300  positives=218  negatives=82

(string-match baseline skipped: y_cov IS the string match, so it would score 1.000)

                     config  k  auroc  delta_vs_6sig
           SWL-A  Lift only  1  0.647         -0.145
     SWL-B  5 cheap signals  5  0.663         -0.130
    SWL-C  6 signals (+CAS)  6  0.793          0.000
SWL-C2 6 signals (cas_mean)  6  0.686         -0.107
          kw_coverage alone  1  0.608         -0.185
                  cas alone  1  0.776         -0.017
                 minus lift  5  0.762         -0.030
            minus relevance  5  0.796          0.004
          minus consistency  5  0.795          0.002
          minus kw_coverage  5  0.796          0.003
              minus clarity  5  0.797          0.005
                  minus cas  5  0.663         -0.130

6-signal weights: {'lift': np.float64(0.776), 'relevance': np.float64(-0.011), 'consistency': np.float64(-0.308), 'kw_coverage': np.float64(0.098), 'clarity': np.fl

## Cell 12 — Abstention operating points

In [15]:
pts = []
for cov in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]:
    t   = np.quantile(full_oof, 1 - cov)
    ans = full_oof >= t
    pts.append(dict(coverage=round(float(ans.mean()), 3),
                    acc_on_answered=round(float(y[ans].mean()), 3) if ans.any() else None,
                    abstention_precision=round(float((1-y[~ans]).mean()), 3) if (~ans).any() else None,
                    n_abstained=int((~ans).sum())))
pts = pd.DataFrame(pts)
print(f"answer everything -> accuracy {y.mean():.3f}\n")
print(pts.to_string(index=False))

answer everything -> accuracy 0.727

 coverage  acc_on_answered  abstention_precision  n_abstained
      1.0            0.727                   NaN            0
      0.9            0.785                 0.800           30
      0.8            0.804                 0.583           60
      0.7            0.852                 0.567           90
      0.6            0.867                 0.483          120
      0.5            0.907                 0.453          150


## Cell 13 — Save

In [16]:
df.drop(columns=['ret_text']).to_csv(CONFIG['out_signals'], index=False)
with open(CONFIG['out_results'], 'w') as f:
    json.dump(dict(config=CONFIG, scope=SCOPE, target=TARGET, n=len(work),
                   corpus_passages=len(passages),
                   base_rates={t: float(df[t].mean()) for t in ['y_cov','y_gen','y_judge']},
                   n_evaluable=int(df['evaluable'].sum()),
                   per_signal=res.to_dict('records'),
                   configs=o.to_dict('records'),
                   swl_weights=dict(zip(ALL6, map(float, W))),
                   operating_points=pts.to_dict('records')), f, indent=2, default=str)
print("saved:", CONFIG['out_signals'])
print("saved:", CONFIG['out_results'])
print("\nIMPORTANT: File > Download > .ipynb WITH OUTPUTS, or these numbers are gone.")
print("None of your eleven earlier notebooks saved its outputs.")

saved: /content/drive/MyDrive/halt_rag_cache/halt_rag_w23_signals.csv
saved: /content/drive/MyDrive/halt_rag_cache/halt_rag_w23_results.json

IMPORTANT: File > Download > .ipynb WITH OUTPUTS, or these numbers are gone.
None of your eleven earlier notebooks saved its outputs.


---

### Still outstanding after this notebook runs

1. **`y_cov` is still the 57%-wrong string match.** Everything measured against
   it is provisional. The real fix is relabelling all 300 rows with the published
   Joren et al. autorater prompt, which also solves the missing second annotator.
2. **`y_judge` uses an invented prompt.** Do not report it until that prompt is
   replaced with the published one.
3. **The corpus is still a pilot**, even with more distractors. The scan reached
   only 120 of your 300 entity pages.
4. **The three-way router (RQ4) is not built.** Your 14 `A_retrieval_failure`
   rows are the ground truth for it.